## **Arabic Named Entity Recognition**
Named entity recognition (NER) is the task of finding spans of text that constitute proper names and labeling the type of such entities.
As we discussed in the lecture, this task can be formulated as a sequence labeling problem, e.g., using the BIO tagging scheme.
Your task is to build a NER tagger for Arabic using Conditional Random Fields (CRFs) and explore different sets of features to achieve improvement over the baseline tagger.
Arabic is a morphologically rich language, which poses certain challenges to NLP.
We use ANERcorp that has 150K words annotated for four entities: Location (LOC), Organization (ORG), and Person (PER), and Miscellaneous (MISC).
Since ANERcorp only has train and test splits, we take 10% of the train as a development set.


## **The Task**
Your task is to improve the feature function (`word2features()`).
Currently, the baseline model only takes into account previous and next words.
You may use external tools to obtain useful features such as POS and morphological tags.
You will also need to submit a report describing your method, result, and analysis (1-2 pages).

## **Data Preparation**

In [8]:
%pip install scikit-learn seqeval
%pip install python-crfsuite
%pip install camel-tools
import sys
!{sys.executable} -m camel_tools.cli.camel_data -i disambig-mle-calima-msa-r13
!test -f ANERcorp-CamelLabSplits.zip || wget https://camel.abudhabi.nyu.edu/anercorp/ANERcorp-CamelLabSplits.zip
!unzip -o ANERcorp-CamelLabSplits.zip -x "__MACOSX/*"

/Users/hajackson/venvs/nlp805/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/Users/hajackson/venvs/nlp805/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/Users/hajackson/venvs/nlp805/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
No new packages will be installed.
zsh:1: command not found: wget
unzip:  cannot find or open ANERcorp-CamelLabSplits.zip, ANERcorp-CamelLabSplits.zip.zip or ANERcorp-CamelLabSplits.zip.ZIP.


In [9]:
from sklearn.model_selection import train_test_split

# read ANER named entity dataset
def read_data(file_path):
    with open(file_path, mode='r') as f:
        sent, sents = [], []
        for line in f.readlines():
            if len(line.strip()) > 0:
                # split the line by space
                token = line.strip().split(' ')
                # unpack the token
                word, ner = token
                # append the tuple (word, ner tag) to a list for one sentence
                sent.append((word, ner.replace('PERS', 'PER')))
            # if the line is empty and the list is not empty
            elif len(sent):
                sents.append(sent)
                sent = []
        if sent:
            sents.append(sent)
    return sents

# read training and test data
_train_sents = read_data('./ANERcorp-CamelLabSplits/ANERCorp_CamelLab_train.txt')
test_sents = read_data('./ANERcorp-CamelLabSplits/ANERCorp_CamelLab_test.txt')

# since this dataset does not have dev set, we take 10% of the train as dev set
train_sents, dev_sents = train_test_split(_train_sents, test_size=0.1, random_state=12345)

## **CAMeL Tools**
CAMeL Tools is an open-source Python Arabic NLP toolkit from CAMeL Lab, NYUAD. I use its MLE disambiguator and CALIMA-Star MSA lexicon to extract morphological features for every token, which are supplied to the CRF `word2features` function to mitigate morphological complexity in Arabic NER.

In [10]:
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.utils.dediac import dediac_ar

mle = MLEDisambiguator.pretrained()

# morphological features kept from the top analysis
MORPH_KEYS = ['pos', 'prc0', 'prc1', 'prc2', 'prc3', 'enc0',
              'gen', 'num', 'asp', 'stt', 'cas']

def add_morph(sents):
    out = []
    for sent in sents:
        analyses = mle.disambiguate([word for word, _ in sent])
        new_sent = []
        for (word, ner), d in zip(sent, analyses):
            a = d.analyses[0].analysis if d.analyses else {}
            morph = {k: a.get(k, 'NA') for k in MORPH_KEYS}
            # lemma without diacritics
            morph['lex'] = dediac_ar(a.get('lex', 'NA'))
            new_sent.append((word, ner, morph))
        out.append(new_sent)
    return out

train_sents = add_morph(train_sents)
dev_sents = add_morph(dev_sents)
test_sents = add_morph(test_sents)
print(train_sents[0][:3])

[('وقال', 'O', {'pos': 'verb', 'prc0': '0', 'prc1': '0', 'prc2': 'wa_conj', 'prc3': '0', 'enc0': '0', 'gen': 'm', 'num': 's', 'asp': 'p', 'stt': 'na', 'cas': 'na', 'lex': 'قال'}), ('إن', 'O', {'pos': 'abbrev', 'prc0': 'na', 'prc1': 'na', 'prc2': 'na', 'prc3': 'na', 'enc0': 'na', 'gen': 'na', 'num': 'na', 'asp': 'na', 'stt': 'na', 'cas': 'na', 'lex': 'إن'}), ('العملية', 'O', {'pos': 'noun', 'prc0': 'Al_det', 'prc1': '0', 'prc2': '0', 'prc3': '0', 'enc0': '0', 'gen': 'f', 'num': 's', 'asp': 'na', 'stt': 'd', 'cas': 'g', 'lex': 'عملية'})]


## **Feature Engineering**
Here, your task is to design a function that tranforms each word into a set of features.
You may want to use existing tools to get POS, morphological features, etc.
The current model gives `0.70` in $F_1$ on the dev set.

add one feature group:
**Morphology (CAMeL Tools)**

POS, lemma, proclitics, enclitic, gender, number, aspect, state, case of the current word; POS of the ±2 neighbours, lemma of the ±1 neighbours, and POS bigrams.

In [11]:
def word2features(sent, i):
    word = sent[i][0]
    morph = sent[i][2]

    features = {
        'bias': 1.0,
        'word': word
    }

    # morphological features of the current word from CAMeL Tools
    for k, v in morph.items():
        features['m_' + k] = v
    features['m_pos|prc0'] = morph['pos'] + '|' + morph['prc0']

    # morphological context: POS of +-2 neighbours, lemma of +-1 neighbours
    for d in (-2, -1, 1, 2):
        j = i + d
        if 0 <= j < len(sent):
            features[f'{d}:m_pos'] = sent[j][2]['pos']
            if abs(d) == 1:
                features[f'{d}:m_lex'] = sent[j][2]['lex']

    # baseline: previous and next word (+ POS bigrams)
    if i > 0:
        prev_word = sent[i-1][0]

        features.update({
            '-1:word': prev_word,
            '-1|0:pos': sent[i-1][2]['pos'] + '|' + morph['pos']
        })
    else:
        features['BOS'] = True

    if i < len(sent)-1:
        next_word = sent[i+1][0]

        features.update({
            '+1:word': next_word,
            '0|+1:pos': morph['pos'] + '|' + sent[i+1][2]['pos']
        })
    else:
        features['EOS'] = True

    return features

Extract the features from the data and train your model.

In [12]:
import pycrfsuite


def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, label, morph in sent]

def sent2tokens(sent):
    return [label for token, label, morph in sent]


X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]

X_dev = [sent2features(s) for s in dev_sents]
y_dev = [sent2labels(s) for s in dev_sents]

# create pycrfsuite.Trainer instance
trainer = pycrfsuite.Trainer(verbose=False)

# load the training data
for xseq, yseq in zip(X_train, y_train):
    trainer.append(xseq, yseq)

# training parameteres
trainer.set_params({
    'c1': 0.05,   # coefficient for L1 penalty
    'c2': 0.05,  # coefficient for L2 penalty
    'max_iterations': 200,  # stop earlier

    # include transitions that are possible, but not observed
    'feature.possible_transitions': True
})

trainer.train('arabic-ner.crfsuite')

## **Evaluation**

Evaluate your model on the development set and the test set.

**Note**: You may run evaluation on the test set **ONLY ONCE** at the end. You should not use the test set to check how well your model is currently performing and modify the model accordingly.
Use the dev set, or create a tune set for this purpose.

In [13]:
# instantiate the trained tagger
tagger = pycrfsuite.Tagger()
tagger.open('arabic-ner.crfsuite')

# tag sentences in the dev set
y_pred_dev = [tagger.tag(xseq) for xseq in X_dev]

In [14]:
from seqeval.metrics import classification_report, f1_score

print('Dev F1: %.4f' % f1_score(y_dev, y_pred_dev))
print(classification_report(y_dev, y_pred_dev, digits=4))

Dev F1: 0.8522
              precision    recall  f1-score   support

         LOC     0.9450    0.9093    0.9268       397
        MISC     0.8500    0.5862    0.6939        87
         ORG     0.8472    0.6971    0.7649       175
         PER     0.8534    0.8216    0.8372       241

   micro avg     0.8949    0.8133    0.8522       900
   macro avg     0.8739    0.7536    0.8057       900
weighted avg     0.8923    0.8133    0.8488       900



## Test

In [15]:
X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]
y_pred_test = [tagger.tag(xseq) for xseq in X_test]

print('Test F1: %.4f' % f1_score(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test, digits=4))

Test F1: 0.6641
              precision    recall  f1-score   support

         LOC     0.8251    0.8166    0.8208       676
        MISC     0.7429    0.4280    0.5431       243
         ORG     0.7347    0.3922    0.5114       459
         PER     0.7651    0.5314    0.6272       907

   micro avg     0.7827    0.5768    0.6641      2285
   macro avg     0.7669    0.5420    0.6256      2285
weighted avg     0.7744    0.5768    0.6523      2285



### Write test predictions
Same format as `ANERCorp_CamelLab_test.pred.txt`

In [16]:
lines = []
for sent, preds in zip(test_sents, y_pred_test):
    lines.append('\n'.join(word + ' ' + tag.replace('PER', 'PERS')
                           for (word, _, _), tag in zip(sent, preds)))

with open('ANERCorp_CamelLab_test.pred.txt', mode='w') as f:
    f.write('\n\n'.join(lines))